# Aim
The aim of this notebook is to prototype an analysis pipeline for Thomas's data. Neurons, cilia all that we can wish for.

## Overview
3D Immunos for neurite, cilia, nuclei, (additionally basal body) - 


# 1. Helper functions

In [ ]:
import limoncello as lc
import pyclesperanto_prototype as cle
from imaris_ims_file_reader.ims import ims
import numpy as np
import os
import re
from tqdm import tqdm
import pyclesperanto_prototype as cle
import matplotlib.pyplot as plt
import napari
from skimage.morphology import skeletonize, medial_axis
from skan.csr import skeleton_to_csgraph, Skeleton, summarize
from skan import draw
from skimage.feature import shape_index
from skimage.measure import regionprops
from scipy.spatial import cKDTree
import pandas as pd
from stackview import insight
from scipy.ndimage import distance_transform_edt

In [6]:

ims_path = r"H:\tlobri\03_Experiments\03_Neurite-Cilia\EXP_E6\EXP_E6_260219\post-hoc_405DAPI_488CG_568PCNT_647TUJ1\E4_CL5g_d5_405DAPI_488CG_561PCNT_647TUJ1_50k_2026-03-06_1_F00.ims"

a, meta = lc.load_image(ims_path)


Opening .ims file: \\idnas37.d.uzh.ch\G_MLS_RB_UHome$\tlobri\03_Experiments\03_Neurite-Cilia\EXP_E6\EXP_E6_260219\post-hoc_405DAPI_488CG_568PCNT_647TUJ1\E4_CL5g_d5_405DAPI_488CG_561PCNT_647TUJ1_50k_2026-03-06_1_F00.ims
Opening readonly file: \\idnas37.d.uzh.ch\G_MLS_RB_UHome$\tlobri\03_Experiments\03_Neurite-Cilia\EXP_E6\EXP_E6_260219\post-hoc_405DAPI_488CG_568PCNT_647TUJ1\E4_CL5g_d5_405DAPI_488CG_561PCNT_647TUJ1_50k_2026-03-06_1_F00.ims 

Shape (T, C, Z, Y, X): (1, 4, 31, 2040, 2040)
Number of timepoints: 1
Number of channels: 4
Image dimensions (Z, Y, X): (31, 2040, 2040)
Z Resolution: 0.484
XY Resolution: (0.305, 0.305)


If for some reason the meta data is not found in the .ims, in many Imaris exports there is a **metadata text file** next to the `.ims` file,
for example:
- `METH-PLTurbID-2h-pfa_2026-02-12_12.33.21.ims`
- `METH-PLTurbID-2h-pfa_2026-02-12_12.33.21_metadata.txt`

This step:
- Looks for a matching `*_metadata.txt` file next to your `.ims`
- Reads voxel sizes (pixel size in µm, Z-step in µm) and channel information
- Stores these values so later steps can report **real volumes in µm³** instead of just voxel counts.

If no metadata file is found, the notebook will keep using default voxel sizes (1.0 µm).

In [7]:
meta_text = lc.load_ims_metadata(ims_path)

No metadata .txt file found next to the .ims file.


watch out as sometimes the format is inconsistent and resolutions are not found in the txt file they will default to 1

Select GPU for clesperanto library

In [8]:
print(cle.available_device_names(dev_type="gpu"))
cle.select_device('NVIDIA GeForce RTX 4090')  # optional but good practice
print(cle.get_device())

['NVIDIA GeForce RTX 4090', 'gfx1036']
<NVIDIA GeForce RTX 4090 on Platform: NVIDIA CUDA (1 refs)>


In [ ]:
# Simple default colormaps for a few channels
DEFAULT_COLORMAPS = ["yellow", "green", "red", "blue", "magenta", "cyan"]
CHANNEL_NAMES = ["cilia", "neurites", "basal_bodies", "nuclei"]
scale = (meta['voxel_size'])
viewer = napari.Viewer(ndisplay=3)
viewer.dims.axis_labels = ['Z', 'Y', 'X']
def set_layer_scale(event):
    layer = event.value
    if hasattr(layer, 'scale'):
        layer.scale = scale

# Connect the function to layer insertion events
viewer.layers.events.inserted.connect(set_layer_scale)

T, C, Z, Y, X = a.shape

for c in range(C):
    #volume = (volumes[c] - volumes[c].min()) / (volumes[c].max() - volumes[c].min())  # min - max normalization 
    volume = a[0,c]
    cmap = DEFAULT_COLORMAPS[c % len(DEFAULT_COLORMAPS)]
    viewer.add_image(volume, name=f"Channel {CHANNEL_NAMES[c]}", colormap=cmap, blending="additive", scale=scale, units='um')

print("Napari viewer opened. Use the sliders to explore. Close the window to continue.")
napari.run()


Napari viewer opened. Use the sliders to explore. Close the window to continue.


adding: gaussian_blur=2
 gaussian_blur=1 difference_of_gaussian=1 laplace_box_of_gaussian_blur=1 sobel_of_gaussian_blur=1  gaussian_blur=2 
adding: difference_of_gaussian=2
 gaussian_blur=1 difference_of_gaussian=1 laplace_box_of_gaussian_blur=1 sobel_of_gaussian_blur=1  gaussian_blur=2  difference_of_gaussian=2 
adding: laplace_box_of_gaussian_blur=2
 gaussian_blur=1 difference_of_gaussian=1 laplace_box_of_gaussian_blur=1 sobel_of_gaussian_blur=1  gaussian_blur=2  difference_of_gaussian=2  laplace_box_of_gaussian_blur=2 
adding: sobel_of_gaussian_blur=2
 gaussian_blur=1 difference_of_gaussian=1 laplace_box_of_gaussian_blur=1 sobel_of_gaussian_blur=1  gaussian_blur=2  difference_of_gaussian=2  laplace_box_of_gaussian_blur=2  sobel_of_gaussian_blur=2 
adding: gaussian_blur=3
 gaussian_blur=1 difference_of_gaussian=1 laplace_box_of_gaussian_blur=1 sobel_of_gaussian_blur=1  gaussian_blur=2  difference_of_gaussian=2  laplace_box_of_gaussian_blur=2  sobel_of_gaussian_blur=2  gaussian_blur=3

c:\Users\qfavey\.conda\envs\napari-env-q\lib\site-packages\apoc\_pixel_classifier.py:391: RuntimeWarning: invalid value encountered in divide
  shares[feature] = counts[feature] / sums


 gaussian_blur=1 difference_of_gaussian=1 laplace_box_of_gaussian_blur=1 sobel_of_gaussian_blur=1  gaussian_blur=2  difference_of_gaussian=2  laplace_box_of_gaussian_blur=2  sobel_of_gaussian_blur=2  gaussian_blur=3  difference_of_gaussian=3  laplace_box_of_gaussian_blur=3  sobel_of_gaussian_blur=3  gaussian_blur=0.5  difference_of_gaussian=0.5  laplace_box_of_gaussian_blur=0.5  sobel_of_gaussian_blur=0.5   difference_of_gaussian=5  laplace_box_of_gaussian_blur=5  sobel_of_gaussian_blur=5  sobel_of_gaussian_blur=4  laplace_box_of_gaussian_blur=4  difference_of_gaussian=4  gaussian_blur=4 
 gaussian_blur=1 difference_of_gaussian=1 laplace_box_of_gaussian_blur=1 sobel_of_gaussian_blur=1  gaussian_blur=2  difference_of_gaussian=2  laplace_box_of_gaussian_blur=2  sobel_of_gaussian_blur=2  gaussian_blur=3  difference_of_gaussian=3  laplace_box_of_gaussian_blur=3  sobel_of_gaussian_blur=3  gaussian_blur=0.5  difference_of_gaussian=0.5  laplace_box_of_gaussian_blur=0.5  sobel_of_gaussian_blur

c:\Users\qfavey\.conda\envs\napari-env-q\lib\site-packages\apoc\_pixel_classifier.py:391: RuntimeWarning: invalid value encountered in divide
  shares[feature] = counts[feature] / sums


 difference_of_gaussian=1 laplace_box_of_gaussian_blur=1 sobel_of_gaussian_blur=1  gaussian_blur=2  difference_of_gaussian=2  laplace_box_of_gaussian_blur=2  sobel_of_gaussian_blur=2  gaussian_blur=3  difference_of_gaussian=3  laplace_box_of_gaussian_blur=3  sobel_of_gaussian_blur=3          sobel_of_gaussian_blur=4  laplace_box_of_gaussian_blur=4  difference_of_gaussian=4  gaussian_blur=4 
adding: gaussian_blur=1
 difference_of_gaussian=1 laplace_box_of_gaussian_blur=1 sobel_of_gaussian_blur=1  gaussian_blur=2  difference_of_gaussian=2  laplace_box_of_gaussian_blur=2  sobel_of_gaussian_blur=2  gaussian_blur=3  difference_of_gaussian=3  laplace_box_of_gaussian_blur=3  sobel_of_gaussian_blur=3          sobel_of_gaussian_blur=4  laplace_box_of_gaussian_blur=4  difference_of_gaussian=4  gaussian_blur=4  gaussian_blur=1 
 difference_of_gaussian=1 laplace_box_of_gaussian_blur=1 sobel_of_gaussian_blur=1  gaussian_blur=2  difference_of_gaussian=2  laplace_box_of_gaussian_blur=2  sobel_of_gaus

c:\Users\qfavey\.conda\envs\napari-env-q\lib\site-packages\apoc\_pixel_classifier.py:391: RuntimeWarning: invalid value encountered in divide
  shares[feature] = counts[feature] / sums


adding: gaussian_blur=1
 difference_of_gaussian=1 sobel_of_gaussian_blur=1  gaussian_blur=2  difference_of_gaussian=2  laplace_box_of_gaussian_blur=2  sobel_of_gaussian_blur=2  gaussian_blur=3  difference_of_gaussian=3  laplace_box_of_gaussian_blur=3  sobel_of_gaussian_blur=3          sobel_of_gaussian_blur=4  laplace_box_of_gaussian_blur=4  difference_of_gaussian=4  gaussian_blur=4  gaussian_blur=1 
adding: laplace_box_of_gaussian_blur=1
 difference_of_gaussian=1 sobel_of_gaussian_blur=1  gaussian_blur=2  difference_of_gaussian=2  laplace_box_of_gaussian_blur=2  sobel_of_gaussian_blur=2  gaussian_blur=3  difference_of_gaussian=3  laplace_box_of_gaussian_blur=3  sobel_of_gaussian_blur=3          sobel_of_gaussian_blur=4  laplace_box_of_gaussian_blur=4  difference_of_gaussian=4  gaussian_blur=4  gaussian_blur=1  laplace_box_of_gaussian_blur=1 
adding: original
 difference_of_gaussian=1 sobel_of_gaussian_blur=1  gaussian_blur=2  difference_of_gaussian=2  laplace_box_of_gaussian_blur=2  s

c:\Users\qfavey\.conda\envs\napari-env-q\lib\site-packages\apoc\_pixel_classifier.py:391: RuntimeWarning: invalid value encountered in divide
  shares[feature] = counts[feature] / sums
